INSTALLING DEPENDENCIES

In [ ]:
print("Installing Dependencies")
!pip install -q langchain
!pip install -q langchain-community
!pip install -q langchain-text-splitters
!pip install -q langchain-huggingface
!pip install -q faiss-cpu
!pip install -q pypdf
!pip install -q sentence-transformers
!pip install -q python-dotenv
!pip install -q langchain-groq

print("Installation Completed")

Installing Dependencies
Installation Completed


DOCUMENTS LOADING

In [4]:
import os

from langchain_community.document_loaders import PyPDFDirectoryLoader 




C:\Users\Abid\AppData\Local\Temp\ipykernel_24912\580968309.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFDirectoryLoader
c:\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
corpus_path = "./zyro-dynamics-hr-corpus/"

loader = PyPDFDirectoryLoader(corpus_path)
documents = loader.load()

print(f"Loaded {len(documents)} documents")

Loaded 39 documents


CHUNKING

In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter (
    chunk_size = 800,
    chunk_overlap = 100
    )

chunks = text_splitter.split_documents(documents) 
print(f"Created {len(chunks)} chunks")

Created 107 chunks


EMBEDDING MODEL

In [9]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model ready")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3606.12it/s]


Embedding model ready


VECTOR DATABASE

In [10]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(
    chunks,
    embedding_model
    )
print("Vector database created")

Vector database created


RETRIEVAL

In [11]:
retriever = vectorstore.as_retriever (
    search_kwargs={"k": 3}
    )
print("Retriever Ready")

Retriever Ready


LLM MODEL

In [29]:
from langchain_groq import ChatGroq
from dotenv import load_dotenv

load_dotenv()

llm_model = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0.4,
    max_tokens=500
)

print("Groq model ready")

Groq model ready


RAG PROMPT

In [15]:
from langchain_core.prompts import ChatPromptTemplate

rag_prompt = ChatPromptTemplate.from_template("""
You are a helpful HR assistant for Zyro Dynamics.
Use only the information provided in the context to answer the employee's question.
Give a clear and direct answer based on the company policy.
Do not guess, assume, or add information that is not in the context.
If the context does not contain enough information to answer the question,
say: "I couldn't find this information in the HR policies provided."

Context:{context}

Employee Question:{question}

Answer:
""")

RAG CHAIN

In [16]:
from langchain_core.output_parsers import StrOutputParser 

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

def rag_chain(question):
    docs = retriever.invoke(question)
    context = format_docs(docs)

    chain = rag_prompt | llm_model | StrOutputParser()

    answer = chain.invoke({"context": context, "question": question})

    return answer


print("Rag chain Ready")

Rag chain Ready


In [20]:
import pandas as pd

test_data = pd.read_csv("test.csv")

print(test_data.shape)
print(test_data.head())

(20, 2)
  question_id                                           question
0         Q01       How does my Earned Leave accrue every month?
1         Q02  How much Earned Leave can I carry forward to n...
2         Q03  How many weeks of Maternity Leave am I entitle...
3         Q04    Do I need a medical certificate for sick leave?
4         Q05  What date does my salary get credited every mo...


In [31]:
answers = []

for question in test_data["question"]:
    answer = rag_chain(question)
    answers.append(answer)

test_data["answer"] = answers

print("All 20 questions completed")

All 20 questions completed
